# VIIRS HTTP Download & Extraction

This notebook demonstrates how to search for VIIRS granules via **earthaccess**, verify HTTPS/S3 URL columns, and extract imagery using **satpy**.

| Property | Value |
|----------|-------|
| 📡 Sensor | VIIRS (NOAA-21) |
| ⏱ Estimated time | ~8 min |
| 💾 Disk usage | ~1 GB |
| 🔐 Auth required | Earthdata 🔐 |

> 📖 New to AER? Read the [main README](../../README.md) for the full quickstart and API overview.

## Setup

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import time

import earthaccess
import geopandas as gpd

from aer.client import AerClient
from aer.interfaces import ExtractionProfile
from aer.search_earthaccess import earthaccess_download_wrapper

## Configuration & Earthdata Authentication

VIIRS L1B granules (NOAA-21) are hosted by NASA and require **Earthdata authentication**.

`earthaccess.login()` handles credential caching (`.netrc` or environment variables). If you have not configured credentials, run `earthaccess.login()` interactively once to store them.

We search two collections simultaneously: `VJ202IMG` (imagery) and `VJ203IMG` (geolocation). `plugin_hints` maps each collection to `search_earthaccess` so AER routes both through the same NASA CMR wrapper.

## Configuration

In [ ]:
# --- Configuration ---
DATE_START = datetime(2025, 12, 10, 0, 0, tzinfo=timezone.utc)
DATE_END = datetime(2025, 12, 20, 0, 0, tzinfo=timezone.utc)

# Load AOI (Buenos Aires province)
geojson_path = Path("../neuquen_city.geojson")
gdf = gpd.read_file(geojson_path)
aoi = gdf.geometry.iloc[0]

# Authenticate with Earthdata
earthaccess.login()

# --- Client Setup ---
client = AerClient()

## ExtractionProfile in Detail

`ExtractionProfile` is an `@attrs.frozen` dataclass defined in `aer.interfaces.core`. Key fields:

1. **`name`** — Arbitrary label for your own bookkeeping.
2. **`resolution`** — Target pixel size in metres. VIIRS I-band native resolution is ~375 m; we target 400 m.
3. **`collection_variables_map`** — Dict mapping each collection to the list of bands/variables you want.
   Here we ask for `I01` from `VJ202IMG` and no variables from `VJ203IMG` (geolocation is handled automatically by the reader).
4. **`reader`**, **`padding`**, **`resampling`**, **`calibration`**, **`satellite`** — Domain-specific extraction parameters.
5. **`extract_params`** — Fallback for any additional plugin-specific key/value pairs.

You can define multiple profiles in one run (e.g. one for day/night band, one for thermal). Each profile produces its own set of tasks and output files.

## Search

In [ ]:
print("Searching...", flush=True)
results = client.search(
    collections=["VJ202IMG", "VJ203IMG"],
    start_datetime=DATE_START,
    end_datetime=DATE_END,
    intersects=aoi,
    plugin_hints={
        "VJ202IMG": "search_earthaccess",
        "VJ203IMG": "search_earthaccess",
    },
)
print(f"Found {len(results)} results", flush=True)
results.head()

## Extraction Parameters & Download Wrapper

`extract_params` is reserved for meta-level or tool-level parameters (e.g. `downloader`).

Because VIIRS data is **not** public S3, we must provide a `downloader`. `earthaccess_download_wrapper` is a thin wrapper around `earthaccess.download()` that conforms to AER's `Downloader` protocol: it accepts `(url, local_path)` and blocks until the file is written. The extractor calls it for every granule URL.

Domain-specific configuration (`padding`, `resampling`, `calibration`, `reader`, `satellite`) lives on the `ExtractionProfile` itself.

`max_batch_workers=2` keeps memory usage moderate for VIIRS (larger granules than MODIS).

## Verify URL columns

In [ ]:
assert "s3_url" in results.columns, "Missing s3_url column"
assert "https_url" in results.columns, "Missing https_url column"
print(f"Columns: {list(results.columns)}", flush=True)
print(f"Sample s3_url: {results['s3_url'].iloc[0]}", flush=True)
print(f"Sample https_url: {results['https_url'].iloc[0]}", flush=True)

## Results & EOIDS Layout

The returned `results_df` is a GeoDataFrame validated against `ArtifactSchema`. Each row represents one extracted GeoTIFF.

The `uri` column contains the absolute path, structured as:

```
extract_buenos_aires_viirs/
  loc-17D20L/
    date-20260401/
      sat-NOAA21/
        loc-17D20L_start-20260401T035400_end-20260401T040000_sat-NOAA21_prod-VJ202IMG+VJ203IMG_band-I01_res-400m.tif
```

This **EOIDS** layout makes it trivial to mosaic, filter by date/cell/satellite, or feed into ML pipelines.

## Prepare Extraction

In [ ]:
# --- Prepare Extraction ---
uri = "extract_neuquen_viirs"

profiles = [
    ExtractionProfile(
        name="viirs_i4",
        resolution=400,
        collection_variables_map={"VJ202IMG": ["I01"], "VJ203IMG": []},
        reader="viirs_l1b",
        padding=2,
        resampling="nearest",
        calibration="reflectance",
        satellite="NOAA21",
    )
]

tasks = client.prepare_for_extraction(
    results,
    target_aoi=aoi,
    uri=uri,
    profiles=profiles,
    target_grid_dist=256000,
    target_grid_overlap=False,
    prepare_params={"cells_per_chunk": 10},
    plugin_hints={"VJ202IMG": "extract_satpy", "VJ203IMG": "extract_satpy"},
)

print(f"Prepared {len(tasks)} extraction tasks", flush=True)

## Extract

In [ ]:
# Clean output directory
uri_path = Path(uri)
if uri_path.exists():
    shutil.rmtree(uri_path)
uri_path.mkdir(parents=True)

print("Extracting...", flush=True)
start_time = time.time()

# extract_params is reserved for meta-level / tool-level parameters.
# Domain-specific config (padding, calibration, reader, etc.) lives on the profile.
extract_params = {
    "downloader": earthaccess_download_wrapper,
}

results_df = client.extract_batches(
    tasks,
    extract_params=extract_params,
    plugin_hints={"VJ202IMG": "extract_satpy", "VJ203IMG": "extract_satpy"},
    max_batch_workers=2,
)

end_time = time.time()
print(f"Extraction took {end_time - start_time:.2f}s")
print(f"Extracted {len(results_df)} artifacts")

## Results

In [ ]:
results_df[["id", "collection", "grid_cell", "uri"]].head()